# Notebook 2: Ventaja Comparativa Revelada (Balassa)

**Economia Internacional — UMET 2026**

---

## Objetivo

En la Clase 2 vimos que **David Ricardo** explica el comercio por diferencias en costos de oportunidad, y que **Bela Balassa** (1965) propuso una forma de medir esas ventajas comparativas en datos reales: el indice de **Ventaja Comparativa Revelada (RCA)**.

En este notebook vamos a:
1. Descargar datos de **composicion de exportaciones** desde la API del Banco Mundial para 10 paises
2. Calcular el **indice RCA de Balassa** por categoria de exportacion
3. Identificar en que se **especializa cada pais** y comparar patrones
4. Discutir si la especializacion de Argentina es una **ventaja genuina** o un **resultado historico**

## Quien fue Bela Balassa?

**Bela Balassa** (1928-1991) fue un economista hungaro-estadounidense. En su paper *"Trade Liberalisation and Revealed Comparative Advantage"* (1965, The Manchester School) propuso una idea simple pero poderosa: si no podemos observar directamente los costos de oportunidad de cada pais (como en el modelo de Ricardo), podemos **inferir** la ventaja comparativa a partir de lo que los paises **efectivamente exportan**.

La logica es: si Argentina exporta proporcionalmente mucho mas alimentos que el promedio mundial, es porque tiene alguna ventaja comparativa en ese sector — ya sea por tecnologia, recursos naturales, clima, o historia.

## Instrucciones

1. **Hacer una copia**: Archivo → Guardar una copia en Drive
2. **Ejecutar las celdas** en orden (Shift+Enter)
3. **Leer las explicaciones** entre las celdas de codigo
4. Al llegar a las celdas de **TAREA**, usar ChatGPT u otra IA para generar el codigo
5. **Escribir sus interpretaciones** en las celdas de texto

> **Importante**: No necesitan saber programar. Lo que importa es que sepan **que preguntarle a la IA** y que puedan **interpretar economicamente** los resultados.

---

## Bloque 1: Descargar datos del Banco Mundial

Vamos a descargar la **composicion de exportaciones** (% de exportaciones de mercancias) para 10 paises, directamente desde la API del Banco Mundial.

Las categorias son:
- **Alimentos**: cereales, carnes, aceites, lacteos, bebidas, tabaco
- **Materias primas agricolas**: algodon, lana, madera, caucho
- **Combustibles**: petroleo, gas, carbon
- **Metales y minerales**: hierro, cobre, aluminio, oro
- **Manufacturas**: maquinaria, quimicos, textiles, vehiculos, electronica

**No hace falta modificar nada aca, solo ejecutar.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import requests

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

COLORES = ['#1F4E79', '#E8833A', '#27AE60', '#E74C3C', '#2E75B6',
           '#0EA5E9', '#9333EA', '#6B7280', '#D97706', '#059669']

print('Librerias cargadas')

In [ ]:
# =============================================================
# DESCARGA DE DATOS DESDE LA API DEL BANCO MUNDIAL
# Composicion de exportaciones de mercancias (% del total)
# =============================================================

PAISES = {
    'ARG': 'Argentina', 'BRA': 'Brasil', 'CHL': 'Chile',
    'CHN': 'China', 'USA': 'Estados Unidos', 'DEU': 'Alemania',
    'KOR': 'Corea del Sur', 'MEX': 'Mexico', 'IND': 'India',
    'WLD': 'Mundo'
}

# Indicadores de composicion de exportaciones del Banco Mundial
CATEGORIAS = {
    'TX.VAL.FOOD.ZS.UN': 'Alimentos',
    'TX.VAL.AGRI.ZS.UN': 'Materias primas agricolas',
    'TX.VAL.FUEL.ZS.UN': 'Combustibles',
    'TX.VAL.MMTL.ZS.UN': 'Metales y minerales',
    'TX.VAL.MANF.ZS.UN': 'Manufacturas',
}

def descargar_composicion(paises_iso, indicador_code, categoria_nombre):
    codigos = ';'.join(paises_iso)
    url = (f'https://api.worldbank.org/v2/country/{codigos}'
           f'/indicator/{indicador_code}?format=json&per_page=5000&date=2015:2022')
    resp = requests.get(url, timeout=30)
    data = resp.json()
    filas = []
    if len(data) > 1 and data[1]:
        for entry in data[1]:
            if entry['value'] is not None:
                iso = entry['countryiso3code']
                filas.append({
                    'pais_iso': iso,
                    'pais': PAISES.get(iso, iso),
                    'categoria': categoria_nombre,
                    'anio': int(entry['date']),
                    'valor': round(float(entry['value']), 2)
                })
    return filas

print('Conectando con la API del Banco Mundial...\n')
todas = []
for cod, nombre in CATEGORIAS.items():
    print(f'  Descargando: {nombre}...')
    filas = descargar_composicion(list(PAISES.keys()), cod, nombre)
    todas.extend(filas)
    print(f'    \u2192 {len(filas)} registros')

df = pd.DataFrame(todas)
print(f'\n\u2713 Descarga completa: {len(df)} registros')
print(f'  Paises: {len(df["pais"].unique())}')
print(f'  Categorias: {len(df["categoria"].unique())}')
print(f'  Periodo: {df["anio"].min()}-{df["anio"].max()}')

In [ ]:
# Tomar el ultimo anio disponible para cada pais y categoria
ultimo = df.sort_values('anio').groupby(['pais', 'pais_iso', 'categoria']).last().reset_index()

# Pivotear: paises en filas, categorias en columnas
comp = ultimo.pivot(index='pais', columns='categoria', values='valor').round(1)
comp = comp[['Alimentos', 'Materias primas agricolas', 'Combustibles', 'Metales y minerales', 'Manufacturas']]

print('Composicion de exportaciones (% del total, ultimo anio disponible):\n')
comp

### Que vemos en esta tabla?

Cada fila muestra como se reparten las exportaciones de un pais entre las 5 categorias. Los numeros son porcentajes del total de exportaciones de mercancias.

Observaciones rapidas:
- **Argentina**: ~60% alimentos — una economia fuertemente especializada en el sector agroalimentario
- **Alemania, Corea, China**: ~80-90% manufacturas — economias industriales
- **Chile**: fuerte en metales y minerales (cobre)
- **Mundo**: el promedio global es la referencia para calcular el RCA

---

## Bloque 2: Calcular el RCA de Balassa

La formula del RCA es:

$$RCA_{pais, categoria} = \frac{\text{Participacion de la categoria en las exportaciones del pais}}{\text{Participacion de la categoria en las exportaciones del mundo}}$$

En nuestros datos, esto es simplemente: **valor del pais / valor del mundo** para cada categoria.

- **RCA > 1**: el pais exporta esa categoria mas que el promedio mundial → esta **especializado**
- **RCA < 1**: el pais exporta esa categoria menos que el promedio mundial → **no especializado**
- **RCA = 1**: exactamente igual al promedio mundial

In [ ]:
# Calcular RCA: valor del pais / valor del mundo
mundo = comp.loc['Mundo']
rca = comp.drop('Mundo').div(mundo).round(2)

print('Indice de Ventaja Comparativa Revelada (RCA de Balassa):\n')
print('RCA > 1 = especializado | RCA < 1 = no especializado\n')
rca

In [ ]:
# Grafico de barras: RCA de cada pais por categoria
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: paises latinoamericanos
latam = ['Argentina', 'Brasil', 'Chile', 'Mexico']
rca.loc[latam].T.plot.bar(ax=axes[0], color=COLORES[:4], edgecolor='white', width=0.7)
axes[0].axhline(y=1, color='black', linewidth=1.5, linestyle='--', label='RCA = 1')
axes[0].set_title('America Latina', fontsize=14, fontweight='bold', color='#1F4E79')
axes[0].set_ylabel('RCA (Balassa)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(fontsize=8)

# Panel derecho: potencias industriales
indust = ['Alemania', 'China', 'Corea del Sur', 'Estados Unidos']
rca.loc[indust].T.plot.bar(ax=axes[1], color=COLORES[4:8], edgecolor='white', width=0.7)
axes[1].axhline(y=1, color='black', linewidth=1.5, linestyle='--', label='RCA = 1')
axes[1].set_title('Potencias industriales', fontsize=14, fontweight='bold', color='#1F4E79')
axes[1].set_ylabel('')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8)

fig.suptitle('Ventaja Comparativa Revelada (RCA de Balassa)', fontsize=15, fontweight='bold', color='#1F4E79', y=1.02)
plt.tight_layout()
plt.show()

### Interpretacion

**America Latina:**
- **Argentina** tiene un RCA altisimo en alimentos (~7): exporta 7 veces mas alimentos (como % de sus exportaciones) que el promedio mundial. Es la especializacion mas fuerte de toda la tabla.
- **Brasil** tambien esta especializado en alimentos (RCA ~5) y algo en materias primas.
- **Chile** se destaca en metales y minerales (el cobre): su RCA puede superar 10 en esa categoria.
- **Mexico** es el mas diversificado de la region: su RCA en manufacturas se acerca a 1 gracias al T-MEC y las maquiladoras.

**Potencias industriales:**
- **Alemania, Corea y China** tienen RCA > 1 en manufacturas — estan especializados en lo industrial.
- **EEUU** es relativamente equilibrado, con RCA cercano a 1 en varias categorias.
- Ninguna potencia industrial tiene RCA > 1 en alimentos — es el espejo de Argentina.

> **Concepto clave**: el RCA revela un patron claro: los paises en desarrollo de America Latina se especializan en recursos naturales y alimentos, mientras las potencias industriales se especializan en manufacturas. Es exactamente lo que Prebisch va a cuestionar en la Unidad 4.

---

## Bloque 3: Argentina en el tiempo

El RCA no es estatico — cambia con el tiempo. Veamos como evoluciono la composicion exportadora de Argentina.

In [ ]:
# Composicion de exportaciones de Argentina en el tiempo
arg = df[df['pais'] == 'Argentina'].pivot(index='anio', columns='categoria', values='valor')
arg = arg[['Alimentos', 'Manufacturas', 'Combustibles', 'Metales y minerales', 'Materias primas agricolas']]

fig, ax = plt.subplots(figsize=(12, 6))
arg.plot.area(ax=ax, stacked=True, alpha=0.8,
              color=['#27AE60', '#2E75B6', '#E74C3C', '#E8833A', '#9333EA'])
ax.set_title('Argentina: composicion de exportaciones (% del total)', 
             fontsize=14, fontweight='bold', color='#1F4E79')
ax.set_ylabel('% de exportaciones de mercancias')
ax.set_xlabel('')
ax.set_ylim(0, 100)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

### Interpretacion

La estructura exportadora de Argentina se mantuvo relativamente estable en los ultimos anos: dominan los **alimentos** (55-65% del total), seguidos por **manufacturas** (~15-20%) y **combustibles** (variable, creciendo con Vaca Muerta).

Esto contrasta con paises como Corea del Sur o China, que transformaron su canasta exportadora de recursos naturales a manufacturas de alta tecnologia en pocas decadas.

> **Pregunta ricardiana**: ¿Argentina se especializa en alimentos porque tiene ventaja comparativa genuina (tierra fertil, clima, tecnologia agricola) o porque no desarrollo ventajas en otros sectores? Esta es la pregunta que los estructuralistas (Prebisch, CEPAL) van a responder diferente que Ricardo.

---

## TAREA

Usando los datos que ya estan cargados (`df`, `comp`, `rca`), resuelvan los siguientes ejercicios.

> **Tip para la IA**: diganle algo como *"Tengo un DataFrame `rca` con paises en filas y categorias de exportacion en columnas (Alimentos, Materias primas agricolas, Combustibles, Metales y minerales, Manufacturas). Los valores son indices RCA de Balassa. Necesito que..."*

---

### Tarea 1: Ranking de especializacion

Para cada **categoria de exportacion**, ordenen los paises de mayor a menor RCA. Muestren el resultado como un grafico de barras horizontales (un panel por categoria). Identifiquen que pais lidera en cada sector.

In [ ]:
# TAREA 1: Escribir el codigo aca
# Pista: usar rca[categoria].sort_values() para cada categoria, graficar con subplots



**Interpretacion Tarea 1** (escribir aca):

1. ¿Que pais lidera en alimentos? ¿Y en manufacturas? ¿Hay algun pais que sea lider en mas de una categoria?

*Respuesta:*

2. ¿Que patron general observan entre paises desarrollados y paises en desarrollo?

*Respuesta:*

---

### Tarea 2: Mexico y el T-MEC

Grafiquen la **evolucion de la composicion de exportaciones de Mexico** en el tiempo (igual que hicimos con Argentina en el Bloque 3). Comparen con Argentina: ¿Mexico se esta industrializando mas rapido?

In [ ]:
# TAREA 2: Escribir el codigo aca
# Pista: filtrar df por pais == 'Mexico', pivotear y graficar area apilada



**Interpretacion Tarea 2** (escribir aca):

1. ¿Que porcentaje de las exportaciones de Mexico son manufacturas? ¿Y de Argentina?

*Respuesta:*

2. ¿Que rol jugo el TLCAN/T-MEC en la transformacion de la estructura exportadora de Mexico? ¿Hay algo equivalente para Argentina?

*Respuesta:*

---

### Tarea 3: ¿Ventaja natural o construida?

Comparen el RCA en **alimentos** de Argentina vs Brasil vs Chile. Los tres son paises sudamericanos con recursos naturales abundantes. Pero ¿tienen el mismo patron de especializacion?

Grafiquen el RCA en alimentos de los tres paises en el tiempo (usando los datos anuales del DataFrame `df`).

In [ ]:
# TAREA 3: Escribir el codigo aca
# Pista: para calcular RCA en el tiempo, necesitan dividir el valor del pais
# por el valor del Mundo para cada anio. Filtrar por categoria 'Alimentos'.



**Interpretacion Tarea 3** (escribir aca):

1. ¿Argentina, Brasil y Chile tienen el mismo nivel de especializacion en alimentos? ¿Que diferencias observan?

*Respuesta:*

2. Si los tres tienen recursos naturales abundantes, ¿por que sus patrones de especializacion son diferentes? (Piensen en: tipo de cambio, politica comercial, historia productiva, acuerdos comerciales)

*Respuesta:*

---

## Reflexion final

Respondan brevemente:

**Ricardo diria que Argentina se especializa en alimentos porque tiene ventaja comparativa en ese sector. ¿Estan de acuerdo? ¿O creen que la especializacion es resultado de la historia, la politica y el tipo de cambio mas que de la "eficiencia natural"?**

*Respuesta:*



---

## Entrega

1. Completar todas las celdas de TAREA
2. Responder todas las preguntas de interpretacion
3. Ejecutar todo el notebook (Menu: Entorno de ejecucion → Ejecutar todo)
4. Descargar como .ipynb (Archivo → Descargar → Descargar .ipynb)
5. Renombrar: `apellido_nombre_NB2.ipynb`
6. Subir al campus de la UMET en la seccion correspondiente

---

*Fuente de datos: Banco Mundial — World Development Indicators, descargados en tiempo real desde https://api.worldbank.org*

*Referencia teorica: Balassa, B. (1965). Trade Liberalisation and Revealed Comparative Advantage. The Manchester School, 33(2), 99-123.*

*Economia Internacional — UMET 2026*